In [ ]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
import shutil
import random
import shutil

## Dataset Reorganization Explanation

This code restructures a chest X-ray pneumonia dataset into new train/validation/test splits.

### 1. **Creates New Folder Structure**
```python
/chest_xray_new/
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```
Uses os.makedirs() to create these directories if they don't exist.

### 2. **Combines and Shuffles Data**
- Merges images from all original splits (train/val/test) for each class (NORMAL/PNEUMONIA)  
- Randomizes the file order to eliminate any original ordering bias

### 3. **Splits Data into New Proportions**
- 80% Training (first 80% of shuffled files)  
- 10% Validation (next 10%)  
- 10% Testing (final 10%)

### 4. **Copies Files to New Structure**
- Uses `shutil.copy()` to populate the new directories with redistributed files

### **Why Do This?**
- Creates fresh splits since original dataset had imbalanced distributions  
- Ensures no data leakage between splits  
- Provides standard 80-10-10 split for experimentation  

⚠️ **Important Note**: This overwrites the original splits - don't use if:  
- You need to preserve the original test set as a true hold-out  
- The dataset already has validated splits for reproducibility

In [ ]:
# ======================================================================
# RESTRUCTURE DATASET
# ======================================================================

dataset_path = '/kaggle/input/chest-xray-pneumonia/chest_xray'
new_dataset_path = '/kaggle/working/chest_xray_new'

# Delete old directory if exists
if os.path.exists(new_dataset_path):
    shutil.rmtree(new_dataset_path)

# Create new directory structure
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        os.makedirs(os.path.join(new_dataset_path, split, cls), exist_ok=True)

# Process classes
for cls in ['NORMAL', 'PNEUMONIA']:
    all_files = []
    
    # Collect files from original splits
    for split in ['train', 'val', 'test']:
        source_folder = os.path.join(dataset_path, split, cls)
        if not os.path.exists(source_folder):
            print(f"Warning: {source_folder} does not exist!")
            continue
            
        files = [os.path.join(source_folder, f) for f in os.listdir(source_folder)]
        all_files.extend(files)
    
    # Shuffle and split
    random.shuffle(all_files)
    train_files = all_files[:int(len(all_files)*0.8)]
    val_files = all_files[int(len(all_files)*0.8):int(len(all_files)*0.9)]
    test_files = all_files[int(len(all_files)*0.9):]
    
    # Copy files
    for src in train_files:
        dest = os.path.join(new_dataset_path, 'train', cls, os.path.basename(src))
        shutil.copy(src, dest)
    
    for src in val_files:
        dest = os.path.join(new_dataset_path, 'val', cls, os.path.basename(src))
        shutil.copy(src, dest)
    
    for src in test_files:
        dest = os.path.join(new_dataset_path, 'test', cls, os.path.basename(src))
        shutil.copy(src, dest)

print("Restructuring complete!")

Restructuring complete!


## Bicubic Interpolation

In [3]:
# ======================================================================
# 1. CONFIGURATION
# ======================================================================
SCALE_FACTOR = 2  # For scale; choose 2, 3, or 4
DATASET_PATH = '/kaggle/working/chest_xray_new'  # Your balanced dataset
LR_PATH = '/kaggle/working/chest_xray_LR'  # For downscaled images
BICUBIC_PATH = '/kaggle/working/chest_xray_bicubic'  # Bicubic upscaled results

In [4]:
# ======================================================================
# 2. CREATE DIRECTORIES
# ======================================================================
for path in [LR_PATH, BICUBIC_PATH]:
    for split in ['train', 'val', 'test']:
        for cls in ['NORMAL', 'PNEUMONIA']:
            os.makedirs(os.path.join(path, split, cls), exist_ok=True)

In [5]:
# ======================================================================
# 3. PROCESSING PIPELINE
# ======================================================================
def process_image(orig_path, lr_path, bicubic_path):
    """Downscale/Upscale and compute metrics"""
    
    # Read image (force grayscale conversion if needed)
    img = cv2.imread(orig_path, cv2.IMREAD_GRAYSCALE)
    
    # Handle odd dimensions for clean scaling
    h, w = img.shape
    h = h - (h % SCALE_FACTOR)
    w = w - (w % SCALE_FACTOR)
    img = cv2.resize(img, (w, h))
    
    # Downscale with bicubic
    lr_img = cv2.resize(img, 
                       (w//SCALE_FACTOR, h//SCALE_FACTOR),
                       interpolation=cv2.INTER_CUBIC)
    
    # Upscale back to original size
    bicubic_img = cv2.resize(lr_img, 
                            (w, h), 
                            interpolation=cv2.INTER_CUBIC)
    
    # Compute metrics
    psnr = cv2.PSNR(img, bicubic_img)
    ssim_val = ssim(img, bicubic_img, 
                   data_range=255,  # For 8-bit images
                   win_size=11,     # Standard for medical images
                   gaussian_weights=True)
    
    # Save processed images
    cv2.imwrite(lr_path, lr_img)
    cv2.imwrite(bicubic_path, bicubic_img)
    
    return psnr, ssim_val

In [6]:
# ======================================================================
# 4. MAIN PROCESSING LOOP
# ======================================================================
metrics = {'psnr': [], 'ssim': []}

for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        src_dir = os.path.join(DATASET_PATH, split, cls)
        
        for img_name in os.listdir(src_dir):
            orig_img_path = os.path.join(src_dir, img_name)
            lr_img_path = os.path.join(LR_PATH, split, cls, img_name)
            bicubic_img_path = os.path.join(BICUBIC_PATH, split, cls, img_name)
            
            psnr, ssim_val = process_image(orig_img_path, lr_img_path, bicubic_img_path)
            
            metrics['psnr'].append(psnr)
            metrics['ssim'].append(ssim_val)

In [7]:
# ======================================================================
# 5. METRIC REPORTING
# ======================================================================
print(f"[Baseline Metrics for Scale Factor {SCALE_FACTOR}X]")
print(f"Average PSNR: {np.mean(metrics['psnr']):.2f} dB")
print(f"Average SSIM: {np.mean(metrics['ssim']):.4f}")
print(f"Processed {len(metrics['psnr'])} images")

[Baseline Metrics for Scale Factor 2X]
Average PSNR: 39.66 dB
Average SSIM: 0.9654
Processed 5856 images


In [8]:
# ======================================================================
# 6. DATA VERIFICATION
# ======================================================================
print("\nSample Verification:")
if len(metrics['psnr']) > 0:  # Only show if images processed
    sample_idx = 0
    orig_img_path = os.path.join(DATASET_PATH, 'train', 'NORMAL', os.listdir(os.path.join(DATASET_PATH, 'train', 'NORMAL'))[sample_idx])
    lr_img_path = os.path.join(LR_PATH, 'train', 'NORMAL', os.listdir(os.path.join(LR_PATH, 'train', 'NORMAL'))[sample_idx])
    print(f"Original Image: {orig_img_path}")
    print(f"LR Image Size: {cv2.imread(lr_img_path).shape}")
else:
    print("No images processed - check directory paths!")


Sample Verification:
Original Image: /kaggle/working/chest_xray_new/train/NORMAL/NORMAL2-IM-0229-0001.jpeg
LR Image Size: (584, 744, 3)
